# Task 5: AI CEO Agent

This notebook builds the AI CEO Agent — the core reasoning system.

The agent's job: take the risks, opportunities, and trends we found in Task 4,
weigh them against each other, prioritize what matters most, and answer:

"If you were the CEO today, what would you do next and why?"

This is ONE agent that reuses retrieval and analysis functions, 
not multiple separate agents. It uses Ollama (Phi-4 Mini) as the open-source LLM.

In [6]:
import json
import os
import requests
import chromadb
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv("../.env")

# Connect to ChromaDB
chroma_client = chromadb.PersistentClient(path="../storage/chromadb")
collection = chroma_client.get_collection("lufthansa_intelligence")

# Load embedding model
model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2",
    token=os.getenv("HF_TOKEN")
)

print(f"Connected to ChromaDB - {collection.count()} documents available")
print("Using Ollama (phi4-mini) as the reasoning engine")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7033.51it/s]


Connected to ChromaDB - 435 documents available
Using Ollama (phi4-mini) as the reasoning engine


## Step 1: Load the Intelligence Results from Task 4

We load the risks, opportunities, and trends we already identified.
The CEO Agent will reason over these rather than rebuilding them.

In [7]:
# Load the saved intelligence results from Task 4
with open("../data/intelligence_results.json", "r", encoding="utf-8") as f:
    intelligence = json.load(f)

risks = intelligence["risks"]
opportunities = intelligence["opportunities"]
trends = intelligence["trends"]

print(f"Risks loaded         : {len(risks)}")
print(f"Opportunities loaded : {len(opportunities)}")
print(f"Trends loaded         : {len(trends)}")

Risks loaded         : 4
Opportunities loaded : 4
Trends loaded         : 4


## Step 2: The CEO Agent Reasoning

We send all risks, opportunities, and trends to the LLM in one prompt.
The LLM acts as the CEO advisor: it reasons, prioritizes, recommends, and justifies.

This directly answers: "If you were the CEO today, what would you do next and why?"

In [8]:
def ceo_agent_recommend(risks, opportunities, trends):

    summary = "RISKS:\n"
    for r in risks:
        summary += f"- {r['title']}: {r['description']}\n"

    summary += "\nOPPORTUNITIES:\n"
    for o in opportunities:
        summary += f"- {o['title']}: {o['description']}\n"

    summary += "\nTRENDS:\n"
    for t in trends:
        summary += f"- {t['title']}: {t['description']}\n"

    prompt = f"""You are the AI Strategic Advisor to the CEO of Lufthansa Group.

{summary}

Write an executive briefing with exactly these 3 sections:

1. WHAT HAPPENED: Summarize the key recent developments (2-3 sentences),
   drawing from the risks, opportunities, and trends above.

2. WHY IT MATTERS: Explain the business significance of these developments
   (2-3 sentences) - what's at stake for Lufthansa.

3. WHAT SHOULD MANAGEMENT DO NEXT: Give your top 3 prioritized actions.
   For each: what to do, why it matters, and which risk/opportunity/trend justifies it.
   Explicitly explain the TRADE-OFF for your top priority - what are you choosing
   NOT to focus on right now, and why is your top choice more urgent than the alternatives?"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "phi4-mini",
            "prompt": prompt,
            "stream": False
        }
    )

    return response.json()["response"]


# Run the CEO Agent
ceo_recommendation = ceo_agent_recommend(risks, opportunities, trends)
print(ceo_recommendation)

1. WHAT HAPPENED: Lufthansa Group has recently navigated through intense competition that led to lower fares but also leveraged opportunities such as extending its long-haul network into new summer routes while integrating AI-powered aircraft turnarounds at Frankfurt Airport.

2. WHY IT MATTERS: These developments are pivotal for Lufthansa's market positioning, aiming towards enhancing operational efficiency and expanding international connectivity which directly impacts revenue streams amidst the challenges of fare wars in a competitive landscape like Europe’s aviation sector.

3. WHAT SHOULD MANAGEMENT DO NEXT:

   1. Focus on Fleet Modernization with Sustainable Aviation Fuel Adoption:
      - Action: Accelerate fleet modernization efforts by integrating sustainable fuels across new aircraft deliveries.
      - Why it Matters & Justification: Investing now reduces future carbon footprint and aligns Lufthansa as a leader in sustainability, tapping into growing eco-conscious consumer 

## Step 3: Save the CEO Recommendation

In [9]:
with open("../data/ceo_recommendation.txt", "w", encoding="utf-8") as f:
    f.write(ceo_recommendation)

print("Saved updated CEO recommendation with trade-off reasoning")

Saved updated CEO recommendation with trade-off reasoning
